# 04 - Estimativa de Economia Pix

Este notebook calcula cenarios hipoteticos de economia potencial com taxas de cartao e tarifas de transferencias tradicionais.

## Observacao Importante

A analise de economia potencial e baseada em cenarios hipoteticos. Os dados publicos nao identificam qual meio de pagamento cada transacao Pix substituiu. Por isso, os valores calculados nao representam economia real comprovada, mas sim uma simulacao analitica para fins educacionais.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name in {"notebooks", "i_notebooks"} else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
from pyspark.sql import functions as F

from src.config import PIX_FEE_SAVINGS_DIR, PIX_MONTHLY_INDICATORS_DIR, PIX_TRANSFER_SAVINGS_DIR, create_project_directories
from src.data_quality import ensure_columns, ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(verbose=False)
spark = get_spark_session("04-fee-savings-pix")

In [ ]:
monthly_df = spark.read.parquet(str(PIX_MONTHLY_INDICATORS_DIR))
ensure_not_empty(monthly_df, "Gold indicadores mensais")
ensure_columns(monthly_df, ["ano_mes", "quantidade_transacoes", "valor_total"], "Gold indicadores mensais")

In [ ]:
card_scenarios = spark.createDataFrame([
    ("Conservador - debito", "MDR debito", 0.10, 0.0108),
    ("Moderado - debito", "MDR debito", 0.20, 0.0108),
    ("Conservador - credito", "MDR credito", 0.10, 0.0226),
    ("Moderado - credito", "MDR credito", 0.20, 0.0226),
], ["cenario", "tipo_taxa", "percentual_substituicao", "taxa_referencia"])

card_savings_df = (
    monthly_df.select(
        "ano_mes",
        "quantidade_transacoes",
        F.col("valor_total").alias("valor_total_pix"),
    )
    .crossJoin(card_scenarios)
    .withColumn("economia_estimada", F.col("valor_total_pix") * F.col("percentual_substituicao") * F.col("taxa_referencia"))
    .withColumn("observacao", F.lit("Cenario hipotetico educacional; nao representa economia real comprovada."))
)

ensure_not_empty(card_savings_df, "Gold economia cartao")

In [ ]:
transfer_scenarios = spark.createDataFrame([
    ("Conservador - transferencia", "tarifa_baixa", 0.05, 5.00),
    ("Conservador - transferencia", "tarifa_media", 0.05, 10.00),
    ("Conservador - transferencia", "tarifa_alta", 0.05, 15.00),
    ("Moderado - transferencia", "tarifa_baixa", 0.10, 5.00),
    ("Moderado - transferencia", "tarifa_media", 0.10, 10.00),
    ("Moderado - transferencia", "tarifa_alta", 0.10, 15.00),
    ("Agressivo - transferencia", "tarifa_baixa", 0.20, 5.00),
    ("Agressivo - transferencia", "tarifa_media", 0.20, 10.00),
    ("Agressivo - transferencia", "tarifa_alta", 0.20, 15.00),
], ["cenario", "tipo_taxa", "percentual_substituicao", "tarifa_referencia"])

transfer_savings_df = (
    monthly_df.select(
        "ano_mes",
        "quantidade_transacoes",
        F.col("valor_total").alias("valor_total_pix"),
    )
    .crossJoin(transfer_scenarios)
    .withColumn("economia_estimada", F.col("quantidade_transacoes") * F.col("percentual_substituicao") * F.col("tarifa_referencia"))
    .withColumn("observacao", F.lit("Cenario hipotetico educacional; nao representa economia real comprovada."))
)

ensure_not_empty(transfer_savings_df, "Gold economia transferencia")

In [ ]:
card_savings_df.printSchema()
card_savings_df.show(10, truncate=False)
transfer_savings_df.printSchema()
transfer_savings_df.show(10, truncate=False)

In [ ]:
card_savings_df.write.mode("overwrite").parquet(str(PIX_FEE_SAVINGS_DIR))
transfer_savings_df.write.mode("overwrite").parquet(str(PIX_TRANSFER_SAVINGS_DIR))
print(f"Economia com cartao gravada em: {PIX_FEE_SAVINGS_DIR.relative_to(PROJECT_DIR)}")
print(f"Economia com transferencias gravada em: {PIX_TRANSFER_SAVINGS_DIR.relative_to(PROJECT_DIR)}")

In [ ]:
spark.stop()